# Comparação de Algoritmos de Machine Learning para Detecção de Alzheimer

## Objetivo
Este notebook realiza uma **comparação completa** de 6 algoritmos de Machine Learning para classificação binária de Alzheimer, cada um em duas versões:

| # | Algoritmo | Descrição |
|---|-----------|------------|
| 1 | **MLP** | Multi-Layer Perceptron (Rede Neural) |
| 2 | **Decision Tree** | Árvore de Decisão |
| 3 | **KNN** | K-Nearest Neighbors |
| 4 | **Logistic Regression** | Regressão Logística |
| 5 | **Random Forest** | Floresta Aleatória |
| 6 | **SVM** | Support Vector Machine |

### Versões Comparadas
- **BASE**: Parâmetros padrão do scikit-learn
- **OTIMIZADO**: Hiperparâmetros ajustados para melhor desempenho

### Métricas de Avaliação
- **Acurácia**: Proporção de acertos totais
- **Precisão**: Proporção de positivos corretos entre os previstos como positivos
- **Recall**: Capacidade de identificar todos os casos positivos (CRUCIAL em diagnóstico médico)
- **F1-Score**: Média harmônica entre precisão e recall
- **AUC-ROC**: Área sob a curva ROC

## 1. Importação das Bibliotecas

In [ ]:
# ==============================================================================
# IMPORTAÇÃO DE BIBLIOTECAS
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
import warnings
warnings.filterwarnings('ignore')

# Sklearn - Modelos
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Sklearn - Pré-processamento e validação
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc,
    roc_auc_score
)

# Balanceamento
from imblearn.over_sampling import SMOTE

# Configuração de estilo
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Seed para reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Bibliotecas importadas com sucesso!")

## 2. Carregamento dos Dados

In [ ]:
# ==============================================================================
# CARREGAMENTO DOS DADOS
# ==============================================================================

import os

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    csv_path = '/content/drive/MyDrive/alzheimers_disease_data.csv'
    print('Rodando no Google Colab')
except ImportError:
    # Tentar caminhos locais comuns
    caminhos = [
        'alzheimers_disease_data.csv',
        './alzheimers_disease_data.csv',
    ]
    csv_path = None
    for caminho in caminhos:
        if os.path.exists(caminho):
            csv_path = caminho
            break
    
    if csv_path is None:
        print('Rodando localmente - Selecione o arquivo CSV')
        import tkinter as tk
        from tkinter import filedialog
        
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        
        csv_path = filedialog.askopenfilename(
            title='Selecione o arquivo alzheimers_disease_data.csv',
            filetypes=[('CSV files', '*.csv'), ('All files', '*.*')]
        )
        root.destroy()
        
        if not csv_path:
            raise FileNotFoundError('Nenhum arquivo selecionado!')

# Carregar dataset
df = pd.read_csv(csv_path)

print(f'Dataset carregado com sucesso!')
print(f'Arquivo: {csv_path}')
print(f'Dimensões: {df.shape[0]} amostras x {df.shape[1]} features')

df.head()

## 3. Pré-processamento dos Dados

In [ ]:
# ==============================================================================
# PRÉ-PROCESSAMENTO
# ==============================================================================

# Remoção de colunas irrelevantes
colunas_remover = ['PatientID', 'DoctorInCharge']
df = df.drop(columns=[col for col in colunas_remover if col in df.columns])
print(f"Colunas removidas: {colunas_remover}")
print(f"Novas dimensões: {df.shape}")

# Verificação de valores nulos
print(f"\nValores nulos: {df.isnull().sum().sum()}")

# Separação features/target
X = df.drop(columns=['Diagnosis'])
y = df['Diagnosis']

print(f"\nFeatures (X): {X.shape}")
print(f"Target (y): {y.shape}")

In [ ]:
# ==============================================================================
# ANÁLISE DA VARIÁVEL ALVO
# ==============================================================================

contagem_classes = y.value_counts()
print("Distribuição da variável alvo:")
print(f"  - Saudável (0): {contagem_classes[0]} ({contagem_classes[0]/len(y)*100:.1f}%)")
print(f"  - Alzheimer (1): {contagem_classes[1]} ({contagem_classes[1]/len(y)*100:.1f}%)")
print(f"  - Razão de desbalanceamento: {contagem_classes[0]/contagem_classes[1]:.2f}:1")

# Gráfico
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71', '#e74c3c']
bars = ax.bar(['Saudável\n(0)', 'Alzheimer\n(1)'], 
              [contagem_classes[0], contagem_classes[1]], 
              color=colors, edgecolor='black', linewidth=1.5)

for bar, count in zip(bars, contagem_classes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, 
            f'{count}\n({count/len(y)*100:.1f}%)', 
            ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('Distribuição do Diagnóstico (ANTES do Balanceamento)', fontweight='bold')
ax.set_ylabel('Número de Pacientes')
ax.set_ylim(0, max(contagem_classes) * 1.3)
plt.tight_layout()
plt.savefig('distribuicao_original.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Balanceamento e Preparação dos Dados

In [ ]:
# ==============================================================================
# BALANCEAMENTO COM SMOTE
# ==============================================================================

print("Aplicando SMOTE para balanceamento...")
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_balanced, y_balanced = smote.fit_resample(X, y)

print(f"\nApós SMOTE:")
print(f"  - Classe 0 (Saudável): {sum(y_balanced == 0)}")
print(f"  - Classe 1 (Alzheimer): {sum(y_balanced == 1)}")
print(f"  - Total: {len(y_balanced)}")

In [ ]:
# ==============================================================================
# DIVISÃO TREINO/TESTE E NORMALIZAÇÃO
# ==============================================================================

# Divisão estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_balanced
)

print("=" * 50)
print("DIVISÃO DOS DADOS")
print("=" * 50)
print(f"Conjunto de treino: {len(X_train)} amostras")
print(f"Conjunto de teste: {len(X_test)} amostras")

# Normalização
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nNormalização aplicada (StandardScaler)")

## 5. Definição dos Algoritmos

### Configurações BASE (Parâmetros Padrão)

In [ ]:
# ==============================================================================
# ALGORITMOS - VERSÃO BASE
# ==============================================================================

algoritmos_base = {
    'MLP': MLPClassifier(
        random_state=RANDOM_STATE,
        max_iter=500
    ),
    'Decision Tree': DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),
    'KNN': KNeighborsClassifier(
        n_neighbors=5
    ),
    'Logistic Regression': LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000
    ),
    'Random Forest': RandomForestClassifier(
        random_state=RANDOM_STATE
    ),
    'SVM': SVC(
        random_state=RANDOM_STATE,
        probability=True
    )
}

print("Algoritmos BASE configurados:")
for nome in algoritmos_base:
    print(f"  - {nome}")

### Configurações OTIMIZADAS

In [ ]:
# ==============================================================================
# ALGORITMOS - VERSÃO OTIMIZADA
# ==============================================================================

algoritmos_otimizados = {
    'MLP': MLPClassifier(
        hidden_layer_sizes=(128, 64, 32),
        activation='relu',
        solver='adam',
        alpha=0.001,
        learning_rate='adaptive',
        learning_rate_init=0.001,
        max_iter=1000,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=RANDOM_STATE
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        criterion='gini',
        random_state=RANDOM_STATE
    ),
    'KNN': KNeighborsClassifier(
        n_neighbors=7,
        weights='distance',
        metric='minkowski',
        p=2
    ),
    'Logistic Regression': LogisticRegression(
        C=1.0,
        penalty='l2',
        solver='lbfgs',
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=RANDOM_STATE
    ),
    'SVM': SVC(
        kernel='rbf',
        C=10,
        gamma='scale',
        probability=True,
        random_state=RANDOM_STATE
    )
}

print("Algoritmos OTIMIZADOS configurados:")
for nome, modelo in algoritmos_otimizados.items():
    print(f"\n  {nome}:")
    params = modelo.get_params()
    # Mostrar apenas parâmetros principais
    params_importantes = {k: v for k, v in params.items() 
                         if not k.startswith('_') and v is not None 
                         and k not in ['verbose', 'n_jobs', 'warm_start']}
    for k, v in list(params_importantes.items())[:5]:
        print(f"    {k}: {v}")

## 6. Função de Treinamento e Avaliação

In [ ]:
# ==============================================================================
# FUNÇÃO DE TREINAMENTO E AVALIAÇÃO
# ==============================================================================

def treinar_e_avaliar(modelo, X_train, X_test, y_train, y_test, nome):
    """Treina um modelo e retorna métricas de avaliação."""
    
    inicio = time()
    modelo.fit(X_train, y_train)
    tempo_treino = time() - inicio
    
    # Predições
    y_pred = modelo.predict(X_test)
    
    # Probabilidades (para ROC)
    if hasattr(modelo, 'predict_proba'):
        y_proba = modelo.predict_proba(X_test)[:, 1]
    else:
        y_proba = modelo.decision_function(X_test)
    
    # Métricas
    metricas = {
        'Algoritmo': nome,
        'Acurácia': accuracy_score(y_test, y_pred),
        'Precisão': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_proba),
        'Tempo (s)': tempo_treino
    }
    
    return metricas, y_pred, y_proba

print("Função de treinamento definida!")

## 7. Treinamento - Modelos BASE

In [ ]:
# ==============================================================================
# TREINAMENTO - MODELOS BASE
# ==============================================================================

print("=" * 70)
print("TREINAMENTO DOS MODELOS BASE")
print("=" * 70)

resultados_base = {}
metricas_base = []

for nome, modelo in algoritmos_base.items():
    print(f"\nTreinando {nome}...", end=" ")
    metricas, y_pred, y_proba = treinar_e_avaliar(
        modelo, X_train_scaled, X_test_scaled, y_train, y_test, nome
    )
    print(f"OK!")
    print(f"  Acurácia: {metricas['Acurácia']:.2%} | F1-Score: {metricas['F1-Score']:.2%} | Recall: {metricas['Recall']:.2%}")
    
    metricas_base.append(metricas)
    resultados_base[nome] = {
        'modelo': modelo,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'metricas': metricas
    }

df_base = pd.DataFrame(metricas_base)
print("\n" + "=" * 70)

## 8. Treinamento - Modelos OTIMIZADOS

In [ ]:
# ==============================================================================
# TREINAMENTO - MODELOS OTIMIZADOS
# ==============================================================================

print("=" * 70)
print("TREINAMENTO DOS MODELOS OTIMIZADOS")
print("=" * 70)

resultados_otim = {}
metricas_otim = []

for nome, modelo in algoritmos_otimizados.items():
    print(f"\nTreinando {nome} (otimizado)...", end=" ")
    metricas, y_pred, y_proba = treinar_e_avaliar(
        modelo, X_train_scaled, X_test_scaled, y_train, y_test, nome
    )
    print(f"OK!")
    print(f"  Acurácia: {metricas['Acurácia']:.2%} | F1-Score: {metricas['F1-Score']:.2%} | Recall: {metricas['Recall']:.2%}")
    
    metricas_otim.append(metricas)
    resultados_otim[nome] = {
        'modelo': modelo,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'metricas': metricas
    }

df_otimizado = pd.DataFrame(metricas_otim)
print("\n" + "=" * 70)

## 9. Resultados Consolidados

In [ ]:
# ==============================================================================
# TABELA DE RESULTADOS - MODELOS BASE
# ==============================================================================

print("\n" + "=" * 80)
print("RESULTADOS - MODELOS BASE")
print("=" * 80)

# Formatação para exibição
df_base_display = df_base.copy()
for col in ['Acurácia', 'Precisão', 'Recall', 'F1-Score', 'AUC-ROC']:
    df_base_display[col] = df_base_display[col].apply(lambda x: f'{x:.4f}')
df_base_display['Tempo (s)'] = df_base_display['Tempo (s)'].apply(lambda x: f'{x:.3f}')

display(df_base_display)

In [ ]:
# ==============================================================================
# TABELA DE RESULTADOS - MODELOS OTIMIZADOS
# ==============================================================================

print("\n" + "=" * 80)
print("RESULTADOS - MODELOS OTIMIZADOS")
print("=" * 80)

df_otim_display = df_otimizado.copy()
for col in ['Acurácia', 'Precisão', 'Recall', 'F1-Score', 'AUC-ROC']:
    df_otim_display[col] = df_otim_display[col].apply(lambda x: f'{x:.4f}')
df_otim_display['Tempo (s)'] = df_otim_display['Tempo (s)'].apply(lambda x: f'{x:.3f}')

display(df_otim_display)

In [ ]:
# ==============================================================================
# MELHORES MODELOS
# ==============================================================================

print("\n" + "=" * 80)
print("MELHORES MODELOS (por F1-Score)")
print("=" * 80)

melhor_base = df_base.loc[df_base['F1-Score'].idxmax()]
melhor_otim = df_otimizado.loc[df_otimizado['F1-Score'].idxmax()]

print(f"\nMelhor modelo BASE: {melhor_base['Algoritmo']}")
print(f"  - F1-Score: {melhor_base['F1-Score']:.4f}")
print(f"  - Acurácia: {melhor_base['Acurácia']:.4f}")
print(f"  - Recall:   {melhor_base['Recall']:.4f}")
print(f"  - AUC-ROC:  {melhor_base['AUC-ROC']:.4f}")

print(f"\nMelhor modelo OTIMIZADO: {melhor_otim['Algoritmo']}")
print(f"  - F1-Score: {melhor_otim['F1-Score']:.4f}")
print(f"  - Acurácia: {melhor_otim['Acurácia']:.4f}")
print(f"  - Recall:   {melhor_otim['Recall']:.4f}")
print(f"  - AUC-ROC:  {melhor_otim['AUC-ROC']:.4f}")

## 10. Visualizações Comparativas

In [ ]:
# ==============================================================================
# GRÁFICO 1: COMPARAÇÃO DE MÉTRICAS
# ==============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metricas = ['Acurácia', 'Precisão', 'Recall', 'F1-Score']
cores = ['#3498db', '#e74c3c']

for idx, metrica in enumerate(metricas):
    ax = axes[idx // 2, idx % 2]
    
    x = np.arange(len(df_base))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, df_base[metrica], width, 
                   label='Base', color=cores[0], edgecolor='black')
    bars2 = ax.bar(x + width/2, df_otimizado[metrica], width,
                   label='Otimizado', color=cores[1], edgecolor='black')
    
    ax.set_xlabel('Algoritmo')
    ax.set_ylabel(metrica)
    ax.set_title(f'Comparação de {metrica}', fontweight='bold', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels(df_base['Algoritmo'], rotation=45, ha='right')
    ax.legend()
    ax.set_ylim(0, 1.15)
    ax.grid(axis='y', alpha=0.3)
    
    # Valores nas barras
    for bar in bars1:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                   xy=(bar.get_x() + bar.get_width()/2, height),
                   xytext=(0, 3), textcoords="offset points",
                   ha='center', va='bottom', fontsize=8)
    
    for bar in bars2:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                   xy=(bar.get_x() + bar.get_width()/2, height),
                   xytext=(0, 3), textcoords="offset points",
                   ha='center', va='bottom', fontsize=8)

plt.suptitle('Comparação de Métricas: Base vs Otimizado', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('comparacao_metricas.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================================================
# GRÁFICO 2: CURVAS ROC
# ==============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Paleta de cores para os algoritmos
cores_alg = plt.cm.Set1(np.linspace(0, 1, 6))

# ROC - Modelos Base
ax = axes[0]
for i, (nome, dados) in enumerate(resultados_base.items()):
    fpr, tpr, _ = roc_curve(y_test, dados['y_proba'])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{nome} (AUC = {roc_auc:.3f})', 
            linewidth=2, color=cores_alg[i])

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aleatório')
ax.set_xlabel('Taxa de Falsos Positivos (FPR)')
ax.set_ylabel('Taxa de Verdadeiros Positivos (TPR)')
ax.set_title('Curvas ROC - Modelos BASE', fontweight='bold', fontsize=14)
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])

# ROC - Modelos Otimizados
ax = axes[1]
for i, (nome, dados) in enumerate(resultados_otim.items()):
    fpr, tpr, _ = roc_curve(y_test, dados['y_proba'])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{nome} (AUC = {roc_auc:.3f})', 
            linewidth=2, color=cores_alg[i])

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aleatório')
ax.set_xlabel('Taxa de Falsos Positivos (FPR)')
ax.set_ylabel('Taxa de Verdadeiros Positivos (TPR)')
ax.set_title('Curvas ROC - Modelos OTIMIZADOS', fontweight='bold', fontsize=14)
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig('curvas_roc_comparacao.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================================================
# GRÁFICO 3: HEATMAP DE MELHORIA PERCENTUAL
# ==============================================================================

metricas_calc = ['Acurácia', 'Precisão', 'Recall', 'F1-Score', 'AUC-ROC']
algoritmos = df_base['Algoritmo'].values

# Calcular melhoria percentual
melhoria = pd.DataFrame(index=algoritmos, columns=metricas_calc)

for metrica in metricas_calc:
    for i, alg in enumerate(algoritmos):
        base = df_base.loc[i, metrica]
        otim = df_otimizado.loc[i, metrica]
        melhoria.loc[alg, metrica] = ((otim - base) / base) * 100 if base > 0 else 0

melhoria = melhoria.astype(float)

plt.figure(figsize=(12, 8))
cmap = sns.diverging_palette(10, 130, as_cmap=True)

ax = sns.heatmap(
    melhoria,
    annot=True,
    fmt='.1f',
    cmap=cmap,
    center=0,
    linewidths=0.5,
    cbar_kws={'label': 'Melhoria (%)'}
)

plt.title('Melhoria Percentual: Base → Otimizado\n(Vermelho = Piorou, Verde = Melhorou)', 
          fontsize=14, fontweight='bold')
plt.xlabel('Métrica')
plt.ylabel('Algoritmo')
plt.tight_layout()
plt.savefig('heatmap_melhoria.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================================================
# GRÁFICO 4: RANKING FINAL (F1-Score)
# ==============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Ranking Base
df_base_sorted = df_base.sort_values('F1-Score', ascending=True)
colors_base = plt.cm.Blues(np.linspace(0.3, 0.9, len(df_base_sorted)))

ax = axes[0]
bars = ax.barh(df_base_sorted['Algoritmo'], df_base_sorted['F1-Score'], 
               color=colors_base, edgecolor='black')
ax.set_xlabel('F1-Score')
ax.set_title('Ranking por F1-Score - Modelos BASE', fontweight='bold', fontsize=14)
ax.set_xlim(0, 1)
ax.grid(axis='x', alpha=0.3)

for bar, val in zip(bars, df_base_sorted['F1-Score']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, 
            f'{val:.3f}', va='center', fontsize=11, fontweight='bold')

# Ranking Otimizado
df_otim_sorted = df_otimizado.sort_values('F1-Score', ascending=True)
colors_otim = plt.cm.Reds(np.linspace(0.3, 0.9, len(df_otim_sorted)))

ax = axes[1]
bars = ax.barh(df_otim_sorted['Algoritmo'], df_otim_sorted['F1-Score'],
               color=colors_otim, edgecolor='black')
ax.set_xlabel('F1-Score')
ax.set_title('Ranking por F1-Score - Modelos OTIMIZADOS', fontweight='bold', fontsize=14)
ax.set_xlim(0, 1)
ax.grid(axis='x', alpha=0.3)

for bar, val in zip(bars, df_otim_sorted['F1-Score']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('ranking_f1score.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================================================
# GRÁFICO 5: MATRIZES DE CONFUSÃO - MODELOS BASE
# ==============================================================================

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (nome, dados) in enumerate(resultados_base.items()):
    ax = axes[idx]
    cm = confusion_matrix(y_test, dados['y_pred'])
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
               xticklabels=['Saudável', 'Alzheimer'],
               yticklabels=['Saudável', 'Alzheimer'],
               annot_kws={'size': 14})
    ax.set_title(f'{nome}', fontweight='bold', fontsize=12)
    ax.set_ylabel('Real')
    ax.set_xlabel('Previsto')

plt.suptitle('Matrizes de Confusão - Modelos BASE', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('matrizes_confusao_base.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================================================
# GRÁFICO 6: MATRIZES DE CONFUSÃO - MODELOS OTIMIZADOS
# ==============================================================================

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (nome, dados) in enumerate(resultados_otim.items()):
    ax = axes[idx]
    cm = confusion_matrix(y_test, dados['y_pred'])
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=ax,
               xticklabels=['Saudável', 'Alzheimer'],
               yticklabels=['Saudável', 'Alzheimer'],
               annot_kws={'size': 14})
    ax.set_title(f'{nome}', fontweight='bold', fontsize=12)
    ax.set_ylabel('Real')
    ax.set_xlabel('Previsto')

plt.suptitle('Matrizes de Confusão - Modelos OTIMIZADOS', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('matrizes_confusao_otimizados.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Salvar Resultados

In [ ]:
# ==============================================================================
# SALVAR RESULTADOS EM CSV
# ==============================================================================

# Salvar DataFrames
df_base.to_csv('resultados_base.csv', index=False)
df_otimizado.to_csv('resultados_otimizados.csv', index=False)

# Criar relatório comparativo
df_base_temp = df_base.copy()
df_base_temp['Versão'] = 'Base'
df_otim_temp = df_otimizado.copy()
df_otim_temp['Versão'] = 'Otimizado'
df_completo = pd.concat([df_base_temp, df_otim_temp], ignore_index=True)
df_completo.to_csv('resultados_completos.csv', index=False)

print("Arquivos salvos:")
print("  - resultados_base.csv")
print("  - resultados_otimizados.csv")
print("  - resultados_completos.csv")
print("  - comparacao_metricas.png")
print("  - curvas_roc_comparacao.png")
print("  - heatmap_melhoria.png")
print("  - ranking_f1score.png")
print("  - matrizes_confusao_base.png")
print("  - matrizes_confusao_otimizados.png")

## 12. Conclusão

In [ ]:
# ==============================================================================
# RESUMO FINAL
# ==============================================================================

print("=" * 80)
print("RESUMO DA COMPARAÇÃO")
print("=" * 80)

print("""
Este notebook comparou 6 algoritmos de Machine Learning para detecção de Alzheimer:

1. MLP (Multi-Layer Perceptron) - Rede Neural
2. Decision Tree - Árvore de Decisão
3. KNN - K-Nearest Neighbors  
4. Logistic Regression - Regressão Logística
5. Random Forest - Floresta Aleatória
6. SVM - Support Vector Machine

Cada algoritmo foi testado em duas versões:
- BASE: Parâmetros padrão do scikit-learn
- OTIMIZADO: Hiperparâmetros ajustados para melhor desempenho

Técnicas utilizadas:
- SMOTE para balanceamento de classes (64.6% vs 35.4% -> 50% vs 50%)
- StandardScaler para normalização
- Divisão estratificada 75/25 (treino/teste)
""")

# Tabela resumo
print("\n" + "-" * 80)
print("TABELA COMPARATIVA FINAL")
print("-" * 80)

resumo = pd.DataFrame({
    'Algoritmo': df_base['Algoritmo'],
    'F1 Base': df_base['F1-Score'].apply(lambda x: f'{x:.4f}'),
    'F1 Otim': df_otimizado['F1-Score'].apply(lambda x: f'{x:.4f}'),
    'Recall Base': df_base['Recall'].apply(lambda x: f'{x:.4f}'),
    'Recall Otim': df_otimizado['Recall'].apply(lambda x: f'{x:.4f}'),
    'AUC Base': df_base['AUC-ROC'].apply(lambda x: f'{x:.4f}'),
    'AUC Otim': df_otimizado['AUC-ROC'].apply(lambda x: f'{x:.4f}')
})

display(resumo)

print("\n" + "=" * 80)
print("COMPARAÇÃO CONCLUÍDA COM SUCESSO!")
print("=" * 80)